# ACME4 Labels Overview

This notebook is intended as a quick start to getting familiar with the labels in the ACME dataset. For this notebook, we'll focus on the PROCESS_UBER_SUMMARY which is a single file with each row representing a process with summarized features for file, network, registry, as well as labels from various sources.

The significant features used in this notebook are:

| Column | Description |
| -- | -- |
| PID_HASH | Unique ID for HOSTNAME+PID+PROCESS_START |
| PROCESS_START | Timestamp for the time process was started |
| PROCESS_NAME | Executable name, without the file path |
| RED_TEAM | 0/1 - Indicates if this process is attributed to a red team action |
| 


The data we'll be using is hosted on: https://gdo-wintap.llnl.gov. The specific files we'll focus on here are:

* PROCESS_UBER_SUMMARY - comprehensive summary of all activity by process.
  * https://gdo168.llnl.gov/data/ACME4/gold/process_uber_summary.parquet
  * 1.77M rows, 296MB

* PROCESS_UBER_SUMMARY split into test/train based on PROCESS_START timestamp
  * 8/19/2024 - 9/8/2024: Test
  * 9/9/2024 - 9/23/2024: Train
  * https://gdo168.llnl.gov/data/ACME4/gold/test-process_uber_summary.parquet
  * https://gdo168.llnl.gov/data/ACME4/gold/train-process_uber_summary.parquet

A complete data dictionary is available at: https://gdo-wintap.llnl.gov.



In [ ]:
import duckdb
%load_ext magic_duckdb
con = duckdb.connect()
%dql -co con

In [ ]:
%dql create or replace view process_uber_summary as from 'data/wintapv6/ACME4/debug/process_uber_summary.parquet'
%dql select if(process_started < '2024-09-09', 'test','train') subset, red_team, count(*) num_processes from process_uber_summary group by all

## Summary by type of label
Here we can see a short, high-level summary by type of label.

For each type, convert to a simple string label and summarize by the permuations. For this view, any non-null value indicates a process that was flagged as "BAD", with the exception of the LOLC flag, which can also indicate "GOOD".


In [ ]:
%%dql
select
	-- Add this to the PUS as a column:
	-- Split into 2 files using 9/9/2024: test/train, in that order...	
	red_team: if(label_num_sources>0 or bad_user is not null,1,0),
	if(label_num_sources>0, 'MANUAL', null) manual,
	bad_user,
	concat_ws('LOLC_', LOLC_CLASS) lolc,
	if(lolbas_num_rows>0, 'LOLBAS', null) lolbas,
	if(mitre_num_rows > 0, 'MITRE', null) mitre,
	if(total_sigma_hits>0, 'SIGMA', null)sigma,
	count(*) num_processes
from process_uber_summary
group by all
order by all


In [ ]:
%%dql
select time_bucket(INTERVAL '1 day',process_started), count(*), list(distinct hostname)
from process_uber_summary
where label_num_sources>0 or bad_user is not null
group by all
order by all

## Manual Labels
Let's dive a little deeper into the MANUAL labels, which are the defined manually by either parsing Caldera after action reports or building small graphs by hand with the red team attacker. Confidence is high for these and false positives are very low.

To help dive in, we'll use the array of source file names found in the LABEL_SOURCES column.

In [ ]:
%%dql 
select label_sources, list_sort(list(distinct hostname)) hosts, count(*) num_processes
from process_uber_summary
group by all
order by all

In [ ]:
%%dql -o bad_users_df
SELECT hostname, user_name, process_name, first_seen, last_seen, mitre_num_rows, lolbas_num_rows, label_num_hits, total_sigma_hits
from process_uber_summary
where bad_user is not null

In [ ]:
import altair as alt
# Add this to the paper as a summary
alt.Chart(bad_users_df).mark_circle(size=20).encode(
    x='first_seen',
    y=alt.Y('user_name'), #,scale=alt.Scale(type="symlog")),
    color='process_name',
    tooltip=['process_name:N','first_seen:T','last_seen:T','hostname']
).properties(
    title='Bad Actors',
    width=1200,
    height=400
).interactive()

In [ ]:
import altair as alt

alt.Chart(bad_users_df).mark_circle(size=20).encode(
    x='first_seen',
    y=alt.Y('mitre_num_rows'), #,scale=alt.Scale(type="symlog")),
    #color='process_name',
    tooltip=['process_name:N','first_seen:T','last_seen:T','hostname']
).properties(
    title='Mitre',
    width=1200,
    height=400
).interactive()

In [ ]:
import altair as alt

alt.Chart(bad_users_df).mark_circle(size=20).encode(
    x='first_seen',
    y=alt.Y('total_sigma_hits'), #,scale=alt.Scale(type="symlog")),
    #color='process_name',
    tooltip=['process_name:N','first_seen:T','last_seen:T','hostname']
).properties(
    title='Sigma',
    width=1200,
    height=400
).interactive()